In [1]:
import numpy

In [2]:
import sentence_transformers

In [3]:
try:
    import unstructured
    print("Step 1: Base unstructured loaded.")
    from unstructured.partition.common import common
    print("Step 2: Partition common loaded.")
    from unstructured.partition.pdf import partition_pdf
    print("Step 3: PDF partitioning loaded.")
except Exception as e:
    print(f"FAILED at: {e}")

Step 1: Base unstructured loaded.
Step 2: Partition common loaded.
Step 3: PDF partitioning loaded.


In [4]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os
import uuid
import json
import hashlib
import pickle
import time
from pathlib import Path
from typing import Dict, Any, List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

# PDF & Document Processing
from unstructured.partition.pdf import partition_pdf
from PIL import Image
from bs4 import BeautifulSoup

# LangChain
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.documents import Document
from langchain.schema.runnable import Runnable
from langchain.prompts import MessagesPlaceholder
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.storage import LocalFileStore
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.retrievers import EnsembleRetriever, BM25Retriever
from langchain_ollama.llms import OllamaLLM
from sentence_transformers import CrossEncoder
import chromadb

print("All imports loaded successfully.")


All imports loaded successfully.


In [ ]:
# ============================================================
# CELL 2: CONFIGURATION + PDF EXTRACTION + CACHING
# ============================================================

# --- Paths are env-overridable; defaults are relative to this notebook ---
PROJECT_ROOT = Path(os.environ.get("MMRAG_ROOT", Path.cwd())).resolve()
PDF_PATH           = os.environ.get("MMRAG_PDF", str(PROJECT_ROOT / "sample.pdf"))
CHROMA_PERSIST_DIR = os.environ.get("MMRAG_CHROMA", str(PROJECT_ROOT / "chromadb"))
CACHE_DIR          = os.environ.get("MMRAG_CACHE",  str(PROJECT_ROOT / "cache"))
DOCSTORE_DIR       = os.environ.get("MMRAG_DOCSTORE", str(PROJECT_ROOT / "docstore"))

OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://127.0.0.1:11434")
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
LLM_MODEL       = "llava:7b"    # vision model, image summaries + image QA
TEXT_LLM_MODEL  = "llama3"      # text model, QA + multi-query + table summary
RERANKER_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Domain is now configurable so the prompt is reusable
DOMAIN_DESCRIPTION = os.environ.get(
    "MMRAG_DOMAIN",
    "the provided document"
)

# Chunking
PARENT_CHUNK_SIZE, PARENT_CHUNK_OVERLAP = 2000, 200
CHILD_CHUNK_SIZE,  CHILD_CHUNK_OVERLAP  = 800, 100

# Retrieval
TOP_K_RETRIEVAL = 10
TOP_K_RERANKED  = 7
BM25_WEIGHT, VECTOR_WEIGHT = 0.4, 0.6
NUM_MULTI_QUERIES = 3
MAX_SUMMARIZATION_WORKERS = 3
MAX_IMAGES_TO_MODEL = 3         # images passed to vision LLM per query

# --- Cache manager ---
class CacheManager:
    """Disk-backed key-value cache. Batched writes via flush()."""
    def __init__(self, cache_dir: str):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.cache_file = self.cache_dir / "summary_cache.json"
        self._dirty = False
        self.cache = self._load()

    def _load(self) -> dict:
        if self.cache_file.exists():
            try:
                return json.loads(self.cache_file.read_text(encoding="utf-8"))
            except (json.JSONDecodeError, IOError):
                return {}
        return {}

    def flush(self):
        if self._dirty:
            self.cache_file.write_text(
                json.dumps(self.cache, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            self._dirty = False

    @staticmethod
    def _hash(content: str) -> str:
        return hashlib.sha256(content.encode("utf-8")).hexdigest()[:24]

    def get(self, content: str, prefix: str = "") -> Optional[str]:
        return self.cache.get(f"{prefix}_{self._hash(content)}")

    def set(self, content: str, value: str, prefix: str = ""):
        self.cache[f"{prefix}_{self._hash(content)}"] = value
        self._dirty = True

cache = CacheManager(CACHE_DIR)
print(f"Cache loaded: {len(cache.cache)} entries.")

# --- PDF extraction ---
retrieved_tables, images, text_parts = [], [], []
print(f"Extracting {os.path.basename(PDF_PATH)}...")
t0 = time.time()

# Note: extract_images_in_pdf is deprecated in new unstructured; use block_types only.
chunks = partition_pdf(
    filename=PDF_PATH,
    extract_images_block_types=["Image"],
    strategy="hi_res",
    extract_image_block_to_payload=True,
    infer_table_structure=True,
    include_metadata=True,
)

MIN_TEXT_LEN = 20
for chunk in chunks:
    ct = str(type(chunk))
    if "Table" in ct:
        if getattr(chunk.metadata, "text_as_html", None):
            retrieved_tables.append(chunk.metadata.text_as_html)
    elif "Image" in ct:
        if getattr(chunk.metadata, "image_base64", None):
            images.append(str(chunk.metadata.image_base64))
    elif any(t in ct for t in ["Text", "Title"]):
        body = chunk.text.strip()
        if len(body) >= MIN_TEXT_LEN:
            text_parts.append(str(chunk))

# Preserve paragraph boundaries so RecursiveCharacterTextSplitter can split on \n\n
texts = "\n\n".join(text_parts)

# Parent-only split here. Children are derived from parents in Cell 4 (no duplication).
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PARENT_CHUNK_SIZE, chunk_overlap=PARENT_CHUNK_OVERLAP
)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE, chunk_overlap=CHILD_CHUNK_OVERLAP
)
parent_texts = parent_splitter.split_text(texts)

print(
    f"Extracted in {time.time()-t0:.1f}s: "
    f"{len(parent_texts)} parents, {len(retrieved_tables)} tables, {len(images)} images"
)


Cache loaded: 0 entries.
Extracting sample.pdf...


No languages specified, defaulting to English.


In [ ]:
# ============================================================
# CELL 3: PARALLEL SUMMARIZATION WITH CACHING
# ============================================================

# Single shared instances — reused across all worker threads
text_llm = OllamaLLM(
    model=TEXT_LLM_MODEL, timeout=90, temperature=0.1,
    num_predict=512, base_url=OLLAMA_BASE_URL,
)
vision_llm_for_summary = OllamaLLM(
    model=LLM_MODEL, timeout=90, temperature=0.1,
    num_predict=512, base_url=OLLAMA_BASE_URL,
)

summary_prompt = PromptTemplate(
    template=(
        "You will be given {input_type}. Summarize it and include all details "
        "and numerical values. No extra commentary.\n\nInput: {value}"
    ),
    input_variables=["input_type", "value"],
)

def summarize_image(idx_img):
    idx, img = idx_img
    # Hash the FULL image bytes, not a prefix (avoids collisions).
    cached = cache.get(img, prefix="img")
    if cached:
        return idx, cached, True
    try:
        summary = vision_llm_for_summary.bind(images=[img]).invoke(
            "Summarize the image in detail; analyze any graphs. "
            "Include numerical values where inferable. No extra commentary."
        )
        cache.set(img, summary, prefix="img")
        return idx, summary, False
    except Exception as e:
        return idx, f"[Image {idx+1} - summary failed: {str(e)[:100]}]", False

def summarize_table(idx_tbl):
    idx, tbl = idx_tbl
    cached = cache.get(tbl, prefix="tbl")
    if cached:
        return idx, cached, True
    try:
        prompt = summary_prompt.invoke({"input_type": "html of tables", "value": tbl})
        summary = text_llm.invoke(prompt)
        cache.set(tbl, summary, prefix="tbl")
        return idx, summary, False
    except Exception as e:
        return idx, f"[Table {idx+1} - summary failed: {str(e)[:100]}]", False

# Parallel image summaries
print(f"Summarizing {len(images)} images (workers={MAX_SUMMARIZATION_WORKERS})...")
t0 = time.time()
image_results = [None] * len(images)
with ThreadPoolExecutor(max_workers=MAX_SUMMARIZATION_WORKERS) as ex:
    futs = {ex.submit(summarize_image, (i, img)): i for i, img in enumerate(images)}
    for fut in as_completed(futs):
        idx, summary, from_cache = fut.result()
        image_results[idx] = summary
        print(f"  Image {idx+1}/{len(images)} ({'cached' if from_cache else 'generated'})")
image_summaries = image_results
img_time = time.time() - t0

# Parallel table summaries
print(f"Summarizing {len(retrieved_tables)} tables (workers={MAX_SUMMARIZATION_WORKERS})...")
t0 = time.time()
table_results = [None] * len(retrieved_tables)
with ThreadPoolExecutor(max_workers=MAX_SUMMARIZATION_WORKERS) as ex:
    futs = {ex.submit(summarize_table, (i, t)): i for i, t in enumerate(retrieved_tables)}
    for fut in as_completed(futs):
        idx, summary, from_cache = fut.result()
        table_results[idx] = summary
        print(f"  Table {idx+1}/{len(retrieved_tables)} ({'cached' if from_cache else 'generated'})")
table_summaries = table_results
tbl_time = time.time() - t0

cache.flush()  # single write at end instead of per-set
print(f"\nSummarization complete: images={img_time:.1f}s, tables={tbl_time:.1f}s")


In [ ]:
# ============================================================
# CELL 4: VECTOR STORE + HYBRID SEARCH
# ============================================================

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

def html_table_to_text(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "html.parser")
        rows = []
        for tr in soup.find_all("tr"):
            cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            rows.append(" | ".join(cells))
        return "\n".join(rows)
    except Exception:
        import re
        return re.sub(r"<[^>]+>", " ", html).strip()

table_plain_texts = [html_table_to_text(t) for t in retrieved_tables]

# Clear both Chroma and docstore together so parent/child IDs stay consistent.
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
if "RAG_chatbot" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("RAG_chatbot")
    print("Cleared stale Chroma collection.")

vectorstore = Chroma(
    collection_name="RAG_chatbot",
    persist_directory=CHROMA_PERSIST_DIR,
    embedding_function=embeddings,
)

# Persistent docstore (LocalFileStore) — survives kernel restarts, stays in sync
# with the persistent Chroma collection.
docstore_path = Path(DOCSTORE_DIR)
if docstore_path.exists():
    # Wipe stale parents that would orphan any re-created vectors.
    for f in docstore_path.glob("*"):
        try:
            f.unlink()
        except OSError:
            pass
docstore_path.mkdir(parents=True, exist_ok=True)

# LocalFileStore stores bytes — wrap Documents with pickle.
class PickleDocStore:
    def __init__(self, path): self.fs = LocalFileStore(path)
    def mset(self, pairs): self.fs.mset([(k, pickle.dumps(v)) for k, v in pairs])
    def mget(self, keys):
        out = []
        for v in self.fs.mget(keys):
            out.append(pickle.loads(v) if v is not None else None)
        return out
    def mdelete(self, keys): self.fs.mdelete(keys)
    def yield_keys(self, prefix=None): yield from self.fs.yield_keys(prefix=prefix)

docstore = PickleDocStore(str(docstore_path))
id_key = "doc_id"

# --- Parents in docstore, children embedded in Chroma ---
parent_ids = [str(uuid.uuid4()) for _ in parent_texts]
parent_docs = [
    Document(page_content=c, metadata={"data_type": "text", "chunk_type": "parent"})
    for c in parent_texts
]
docstore.mset(list(zip(parent_ids, parent_docs)))

all_child_docs = []
child_plain_texts = []  # kept for BM25 (single granularity)
for p_idx, parent_content in enumerate(parent_texts):
    for child_content in child_splitter.split_text(parent_content):
        all_child_docs.append(Document(
            page_content=child_content,
            metadata={id_key: parent_ids[p_idx], "data_type": "text", "chunk_type": "child"},
        ))
        child_plain_texts.append(child_content)
vectorstore.add_documents(all_child_docs)

# --- Tables: raw HTML in docstore, summary + plain text embedded ---
table_ids = [str(uuid.uuid4()) for _ in table_summaries]
table_parent_docs = [
    Document(page_content=t, metadata={"data_type": "table", "table_plain": table_plain_texts[i]})
    for i, t in enumerate(retrieved_tables)
]
docstore.mset(list(zip(table_ids, table_parent_docs)))

table_search_docs = []
for i, summary in enumerate(table_summaries):
    table_search_docs.append(Document(
        page_content=summary,
        metadata={id_key: table_ids[i], "data_type": "table_sum"},
    ))
    table_search_docs.append(Document(
        page_content=table_plain_texts[i],
        metadata={id_key: table_ids[i], "data_type": "table_text"},
    ))
vectorstore.add_documents(table_search_docs)

# --- Images: base64 in docstore, summary embedded ---
img_ids = [str(uuid.uuid4()) for _ in images]
img_parent_docs = [
    Document(page_content=img, metadata={"data_type": "image"}) for img in images
]
docstore.mset(list(zip(img_ids, img_parent_docs)))

image_summary_docs = [
    Document(page_content=s, metadata={id_key: img_ids[i], "data_type": "image_sum"})
    for i, s in enumerate(image_summaries)
]
vectorstore.add_documents(image_summary_docs)

# --- Retrievers ---
multi_vector_retriever = MultiVectorRetriever(
    vectorstore=vectorstore, docstore=docstore, id_key=id_key,
    search_kwargs={"k": TOP_K_RETRIEVAL},
)

# BM25 over CHILDREN ONLY + one table representation + image summaries.
# No duplication of parent vs child content.
bm25_docs = (
    [Document(page_content=c, metadata={"data_type": "text"}) for c in child_plain_texts]
    + [Document(page_content=t, metadata={"data_type": "table_text"}) for t in table_plain_texts]
    + [Document(page_content=s, metadata={"data_type": "image_sum"}) for s in image_summaries]
)
bm25_retriever = BM25Retriever.from_documents(bm25_docs, k=TOP_K_RETRIEVAL)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K_RETRIEVAL})
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[BM25_WEIGHT, VECTOR_WEIGHT],
)

# Reverse maps so the QA chain can resolve base64/HTML back to its id/summary in O(1)
image_id_map      = {img_ids[i]: images[i] for i in range(len(images))}
image_b64_to_id   = {images[i]: img_ids[i] for i in range(len(images))}
image_summary_map = {img_ids[i]: image_summaries[i] for i in range(len(image_summaries))}
table_id_map      = {table_ids[i]: table_plain_texts[i] for i in range(len(table_plain_texts))}

print(
    f"Indexed: {len(all_child_docs)} child text docs, "
    f"{len(table_search_docs)} table docs, {len(image_summary_docs)} image summaries"
)
print(f"BM25 index: {len(bm25_docs)} docs (children + tables + image summaries, no dup)")
print(f"Hybrid retriever ready (BM25={BM25_WEIGHT}, Vector={VECTOR_WEIGHT})")


In [ ]:
from bs4 import BeautifulSoup
from tabulate import tabulate

def print_html_table(html):
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")

    if not tables:
        print("No tables found in the HTML.")
        return

    for index, table in enumerate(tables, start=1):
        headers = []
        rows = []

        # Get headers if available
        header_row = table.find("tr")
        if header_row:
            headers = [th.get_text(strip=True) for th in header_row.find_all("th")]

        # Get all rows (skip header if already taken)
        for tr in table.find_all("tr")[1 if headers else 0:]:
            row = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            rows.append(row)

        print(f"\nTable {index}:")
        print(tabulate(rows, headers=headers if headers else "firstrow", tablefmt="grid"))

import base64
from io import BytesIO
from PIL import Image

def show_image_from_base64(base64_str):
    try:
        # Decode base64 string
        image_data = base64.b64decode(base64_str)
        image = Image.open(BytesIO(image_data))

        # Show the image
        image.show()

    except Exception as e:
        print("Error displaying image:", e)

In [ ]:
# ============================================================
# CELL 6: ADVANCED QA CHAIN
# ============================================================

class CrossEncoderReranker:
    """Cross-encoder reranker; image docs bypass scoring but are retained."""
    def __init__(self, model_name: str = RERANKER_MODEL, top_k: int = TOP_K_RERANKED):
        print(f"Loading re-ranker: {model_name}...")
        self.model = CrossEncoder(model_name)
        self.top_k = top_k

    def rerank(self, query: str, docs: List[Document]) -> List[Document]:
        if not docs:
            return docs
        scorable, image_docs = [], []
        for doc in docs:
            dt = doc.metadata.get("data_type", "text") if hasattr(doc, "metadata") else "text"
            (image_docs if dt == "image" else scorable).append(doc)
        if not scorable:
            return image_docs[:self.top_k]
        pairs = [(query, d.page_content[:1000]) for d in scorable]
        scores = self.model.predict(pairs)
        ranked = [d for d, _ in sorted(zip(scorable, scores), key=lambda x: x[1], reverse=True)]
        return ranked[:self.top_k] + image_docs[:2]

reranker = CrossEncoderReranker()

class MultiQueryGenerator:
    def __init__(self, llm, num_queries: int = NUM_MULTI_QUERIES):
        self.llm = llm
        self.num_queries = num_queries
        self.prompt = PromptTemplate(
            template=(
                "Generate {num_queries} alternative phrasings of the question below. "
                "Use synonyms, rephrase technically, or target different angles. "
                "Output ONLY the questions, one per line. No numbering.\n\n"
                "Original: {question}"
            ),
            input_variables=["question", "num_queries"],
        )

    def generate_queries(self, question: str) -> List[str]:
        queries = [question]
        try:
            response = self.llm.invoke(self.prompt.invoke(
                {"question": question, "num_queries": self.num_queries}
            ))
            generated = [q.strip() for q in response.strip().split("\n")
                         if q.strip() and len(q.strip()) > 10]
            queries.extend(generated[:self.num_queries])
        except Exception as e:
            print(f"  Multi-query generation failed: {e}. Using original only.")
        return queries

multi_query_gen = MultiQueryGenerator(text_llm, NUM_MULTI_QUERIES)

# Vision LLM used at query time for image-bearing contexts
vision_llm = OllamaLLM(
    model=LLM_MODEL, timeout=120, temperature=0.1,
    num_predict=512, base_url=OLLAMA_BASE_URL,
)

def _full_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

class AdvancedMultimodalQAChain(Runnable):
    def __init__(self, hybrid_retriever, multi_vector_retriever, reranker,
                 multi_query_gen, llm, vision_llm, qa_prompt,
                 image_id_map=None, table_id_map=None,
                 image_summary_map=None, image_b64_to_id=None):
        self.hybrid_retriever = hybrid_retriever
        self.multi_vector_retriever = multi_vector_retriever
        self.reranker = reranker
        self.multi_query_gen = multi_query_gen
        self.llm = llm
        self.vision_llm = vision_llm
        self.qa_prompt = qa_prompt
        self.image_id_map = image_id_map or {}
        self.table_id_map = table_id_map or {}
        self.image_summary_map = image_summary_map or {}
        self.image_b64_to_id = image_b64_to_id or {}

    def _retrieve_and_rerank(self, question, queries, verbose=False):
        all_docs, seen = [], set()
        for q in queries:
            for source in ("hybrid", "mv"):
                try:
                    docs = (self.hybrid_retriever if source == "hybrid"
                            else self.multi_vector_retriever).invoke(q)
                except Exception:
                    continue
                for doc in docs:
                    # FULL-content hash (not [:500]) to avoid overlap-prefix collisions
                    h = _full_hash(doc.page_content)
                    if h not in seen:
                        seen.add(h)
                        all_docs.append(doc)
        if verbose:
            counts = {}
            for d in all_docs:
                counts[d.metadata.get("data_type", "?")] = counts.get(d.metadata.get("data_type", "?"), 0) + 1
            print(f"  [Retrieval] {len(all_docs)} unique: {counts}")
        reranked = self.reranker.rerank(question, all_docs)
        if verbose:
            counts = {}
            for d in reranked:
                counts[d.metadata.get("data_type", "?")] = counts.get(d.metadata.get("data_type", "?"), 0) + 1
            print(f"  [Re-ranked] {len(reranked)}: {counts}")
        return reranked

    def _build_multimodal_context(self, docs):
        texts, tables, image_summaries_out, image_b64_list = [], [], [], []
        for doc in docs:
            content = str(doc.page_content)
            dt = doc.metadata.get("data_type", "text") if hasattr(doc, "metadata") else "text"
            if dt == "text":
                texts.append(content)
            elif dt == "table":
                plain = doc.metadata.get("table_plain", "")
                tables.append(plain if plain else content)
            elif dt == "table_sum":
                texts.append(f"[Table Summary]: {content}")
                doc_id = doc.metadata.get("doc_id", "")
                if doc_id and doc_id in self.table_id_map:
                    tables.append(self.table_id_map[doc_id])
            elif dt == "table_text":
                tables.append(content)
            elif dt == "image":
                image_b64_list.append(content)
                # O(1) reverse lookup instead of linear scan
                img_id = self.image_b64_to_id.get(content)
                if img_id and img_id in self.image_summary_map:
                    image_summaries_out.append(self.image_summary_map[img_id])
            elif dt == "image_sum":
                image_summaries_out.append(content)
                doc_id = doc.metadata.get("doc_id", "")
                if doc_id and doc_id in self.image_id_map:
                    image_b64_list.append(self.image_id_map[doc_id])

        parts = []
        if texts:              parts.append("[Text Content]:\n" + "\n---\n".join(texts))
        if tables:             parts.append("[Table Content]:\n" + "\n---\n".join(tables))
        if image_summaries_out: parts.append("[Image Descriptions]:\n" + "\n---\n".join(image_summaries_out))
        context = "\n\n".join(parts) if parts else "No relevant context found."
        return context, image_b64_list

    def _run(self, input_data, stream=False):
        question = input_data["question"]
        chat_history = input_data.get("chat_history", [])
        verbose = input_data.get("verbose", False)

        queries = self.multi_query_gen.generate_queries(question)
        reranked = self._retrieve_and_rerank(question, queries, verbose=verbose)
        text_context, image_b64_list = self._build_multimodal_context(reranked)

        messages = self.qa_prompt.format_messages(
            question=question, context=text_context, chat_history=chat_history,
        )

        if image_b64_list:
            try:
                active = self.vision_llm.bind(images=image_b64_list[:MAX_IMAGES_TO_MODEL])
                if len(image_b64_list) > MAX_IMAGES_TO_MODEL:
                    print(f"  [Note] {len(image_b64_list) - MAX_IMAGES_TO_MODEL} "
                          f"image(s) omitted (cap={MAX_IMAGES_TO_MODEL}).")
            except Exception:
                active = self.llm
        else:
            active = self.llm

        if stream:
            return active.stream(messages)
        return active.invoke(messages)

    def invoke(self, input_data, config=None):
        return self._run(input_data, stream=False)

    def stream_response(self, input_data):
        full = ""
        for chunk in self._run(input_data, stream=True):
            print(chunk, end="", flush=True)
            full += chunk
        print()
        return full

    # Back-compat helpers
    def parse_docs(self, docs):
        ctx, imgs = self._build_multimodal_context(docs)
        return {"text_context": ctx, "images": imgs}
    def build_text_context(self, d):
        return d.get("text_context", "")

print("QA chain components ready.")


In [ ]:
# ============================================================
# CELL 7: PROMPT + DEMO QUERY
# ============================================================

# Context goes in the FIRST system message so chat models weight it highest.
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     f"You are a knowledgeable technical assistant specializing in {DOMAIN_DESCRIPTION}. "
     "Answer using the provided context, which may include text, tables, and image "
     "descriptions. Synthesize information from ALL context types. Include specific "
     "numerical values, equations, and technical details when available. "
     "If context is missing, provide what you can and flag the gap.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

qa_chain = AdvancedMultimodalQAChain(
    hybrid_retriever=hybrid_retriever,
    multi_vector_retriever=multi_vector_retriever,
    reranker=reranker,
    multi_query_gen=multi_query_gen,
    llm=text_llm,
    vision_llm=vision_llm,
    qa_prompt=qa_prompt,
    image_id_map=image_id_map,
    table_id_map=table_id_map,
    image_summary_map=image_summary_map,
    image_b64_to_id=image_b64_to_id,
)

# Chat history persists across cell executions if you re-run this cell.
chat_history: List[Any] = globals().get("chat_history", [])

def ask(question: str, stream: bool = True, verbose: bool = False) -> str:
    """One-shot Q&A helper. Call repeatedly from new cells instead of a blocking loop."""
    print(f"\nYou: {question}")
    t0 = time.time()
    if stream:
        print("Assistant: ", end="", flush=True)
        answer = qa_chain.stream_response(
            {"question": question, "chat_history": chat_history, "verbose": verbose}
        )
    else:
        answer = qa_chain.invoke(
            {"question": question, "chat_history": chat_history, "verbose": verbose}
        )
        print(f"Assistant: {answer}")
    print(f"  [{time.time()-t0:.1f}s]")
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    return answer

# Example (edit or add new cells calling ask("...")):
# ask("What is the method of joints?")
print("QA chain ready. Use ask('your question') to query.")


In [ ]:
# ============================================================
# CELL 8: EVALUATION FRAMEWORK (LLM judge + deterministic retrieval hit-rate)
# ============================================================

class RAGEvaluator:
    def __init__(self, qa_chain, judge_llm):
        self.qa_chain = qa_chain
        self.llm = judge_llm

    @staticmethod
    def _extract_score(text: str) -> float:
        import re
        nums = re.findall(r"\b(\d+(?:\.\d+)?)\b", text.strip())
        for n in nums:
            v = float(n)
            if 0 <= v <= 10:
                return min(v / 10.0, 1.0)
        return 0.5

    def retrieval_hit_rate(self, retrieved: List[Document], expected_keywords: List[str]) -> float:
        """Deterministic: fraction of expected keywords that appear in any retrieved doc."""
        if not expected_keywords:
            return float("nan")
        hay = " ".join(d.page_content.lower() for d in retrieved
                       if d.metadata.get("data_type") != "image")
        hits = sum(1 for kw in expected_keywords if kw.lower() in hay)
        return hits / len(expected_keywords)

    def _judge(self, template, **kwargs) -> float:
        p = PromptTemplate(template=template, input_variables=list(kwargs.keys()))
        try:
            return self._extract_score(self.llm.invoke(p.invoke(kwargs)))
        except Exception:
            return 0.5

    def evaluate(self, question: str, retrieved: List[Document], answer: str, context: str,
                 expected_keywords: Optional[List[str]] = None) -> dict:
        faithfulness = self._judge(
            "Rate 0-10 how faithful this answer is to the context (no fabrication).\n"
            "Context: {context}\nQ: {q}\nA: {a}\nScore:",
            context=context[:2000], q=question, a=answer[:500],
        )
        relevance = self._judge(
            "Rate 0-10 how well this answer addresses the question.\n"
            "Q: {q}\nA: {a}\nScore:",
            q=question, a=answer[:500],
        )
        out = {
            "answer_faithfulness": round(faithfulness, 2),
            "answer_relevance": round(relevance, 2),
        }
        if expected_keywords is not None:
            out["retrieval_hit_rate"] = round(self.retrieval_hit_rate(retrieved, expected_keywords), 2)
        return out

    def run(self, test_cases: List[dict]) -> dict:
        """test_cases: [{'question': str, 'expected_keywords': [str, ...]}] (keywords optional)."""
        results = []
        print(f"Running evaluation on {len(test_cases)} cases...\n")
        for i, case in enumerate(test_cases):
            q = case["question"]
            kws = case.get("expected_keywords")
            print(f"  [{i+1}/{len(test_cases)}] {q[:70]}")
            try:
                queries = self.qa_chain.multi_query_gen.generate_queries(q)
                reranked = self.qa_chain._retrieve_and_rerank(q, queries)
                context, _ = self.qa_chain._build_multimodal_context(reranked)
                answer = self.qa_chain.invoke({"question": q, "chat_history": []})
                scores = self.evaluate(q, reranked, answer, context, kws)
                row = {"question": q, "answer": answer[:200],
                       "num_docs": len(reranked), **scores}
                results.append(row)
                print(f"    {scores}")
            except Exception as e:
                print(f"    FAILED: {e}")
                results.append({"question": q, "error": str(e)})

        valid = [r for r in results if "answer_faithfulness" in r]
        agg = {}
        if valid:
            for k in ("answer_faithfulness", "answer_relevance", "retrieval_hit_rate"):
                vals = [r[k] for r in valid if k in r and isinstance(r[k], (int, float))]
                if vals:
                    agg[f"avg_{k}"] = round(sum(vals) / len(vals), 2)
        print(f"\n{'='*50}\nEVALUATION SUMMARY ({len(valid)}/{len(test_cases)})")
        for k, v in agg.items():
            print(f"  {k}: {v}")
        print("=" * 50)
        return {"individual": results, "aggregate": agg}

# Stronger judge config: larger num_predict so scoring isn't cut off.
eval_llm = OllamaLLM(
    model=TEXT_LLM_MODEL, timeout=90, temperature=0.0,
    num_predict=128, base_url=OLLAMA_BASE_URL,
)
evaluator = RAGEvaluator(qa_chain, eval_llm)

# Example: supply expected keywords per question for deterministic retrieval scoring.
test_cases = [
    {"question": "What is the method of joints?",
     "expected_keywords": ["joint", "equilibrium", "truss"]},
    {"question": "How do you calculate reactions at supports?",
     "expected_keywords": ["reaction", "support", "equilibrium"]},
    {"question": "What is the difference between a truss and a frame?",
     "expected_keywords": ["truss", "frame", "member"]},
    {"question": "Explain the principle of superposition in structural analysis.",
     "expected_keywords": ["superposition", "linear", "load"]},
    {"question": "What are the types of loads on a structure?",
     "expected_keywords": ["dead", "live", "load"]},
]
# Uncomment to run:
# eval_results = evaluator.run(test_cases)

print("Evaluator ready. Call evaluator.run(test_cases) to execute.")
